# 03 - Gold Layer

Create business-ready tables and aggregations.
- Fact tables (pre-joined core business events)
- Aggregated summary tables (daily/monthly metrics)
- Write to `workspace.gold` schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_orders AS
SELECT 
    o.order_id,
    i.order_item_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_timestamp,
    CAST(o.order_purchase_timestamp AS DATE) AS order_purchase_date,
    o.order_approved_at,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    c.customer_city,
    c.customer_state,
    i.product_id,
    i.seller_id,
    INITCAP(REPLACE(p.product_category_name_english, '_', ' ')) AS product_category,
    i.price,
    i.freight_value,
    DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) AS delivery_days,
    CASE 
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN true 
        ELSE false 
    END AS is_late_delivery
FROM workspace.silver.orders o
JOIN workspace.silver.customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN workspace.silver.order_items i 
    ON o.order_id = i.order_id
LEFT JOIN workspace.silver.products p 
    ON i.product_id = p.product_id;

In [0]:
%sql
SELECT * FROM workspace.gold.fact_orders LIMIT 10;

##Create a Daily Sales Summary (Aggregation)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.daily_sales_summary AS
SELECT 
    CAST(order_purchase_timestamp AS DATE) AS sales_date,
    product_category,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS total_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight
FROM workspace.gold.fact_orders
WHERE product_category IS NOT NULL
GROUP BY 
    CAST(order_purchase_timestamp AS DATE),
    product_category
ORDER BY sales_date DESC;

In [0]:
%sql
SELECT * FROM workspace.gold.daily_sales_summary LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_date AS
SELECT
    date AS date_key,
    YEAR(date) AS year,
    MONTH(date) AS month_number,
    DATE_FORMAT(date, 'MMMM') AS month_name,
    QUARTER(date) AS quarter,
    DATE_FORMAT(date, 'EEEE') AS day_of_week,
    DAYOFWEEK(date) AS day_number,
    CASE WHEN DAYOFWEEK(date) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM (
    SELECT EXPLODE(SEQUENCE(
        DATE('2016-01-01'), 
        DATE('2018-12-31'), 
        INTERVAL 1 DAY
    )) AS date
);

In [0]:
%sql
SELECT * FROM workspace.gold.dim_date LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_payments AS
SELECT
    p.order_id,
    p.payment_sequential,
    o.customer_id,
    o.order_status,
    INITCAP(REPLACE(p.payment_type, '_', ' ')) AS payment_type,
    p.payment_installments,
    p.payment_value,
    o.order_purchase_timestamp,
    CAST(o.order_purchase_timestamp AS DATE) AS order_purchase_date,
    c.customer_state
FROM workspace.silver.order_payments p
JOIN workspace.silver.orders o
    ON p.order_id = o.order_id
JOIN workspace.silver.customers c
    ON o.customer_id = c.customer_id;

In [0]:
%sql
SELECT * FROM workspace.gold.fact_payments LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_sellers AS
SELECT DISTINCT
    s.seller_id,
    INITCAP(s.seller_city) AS seller_city,
    s.seller_state
FROM workspace.silver.sellers s;

In [0]:
%sql
SELECT * FROM workspace.gold.dim_sellers LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_customer AS
SELECT DISTINCT
    customer_unique_id,
    FIRST(customer_city) AS customer_city,
    FIRST(customer_state) AS customer_state
FROM workspace.silver.customers
GROUP BY customer_unique_id;

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customer LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_product AS
SELECT DISTINCT
    product_id,
    INITCAP(REPLACE(product_category_name_english, '_', ' ')) AS product_category,
    product_category_name AS product_category_portuguese,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM workspace.silver.products;

In [0]:
%sql
SELECT 
    'fact_orders' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_orders FROM workspace.gold.fact_orders
UNION ALL
SELECT 
    'fact_payments', COUNT(*), COUNT(DISTINCT order_id) FROM workspace.gold.fact_payments
UNION ALL
SELECT 
    'dim_customer', COUNT(*), NULL FROM workspace.gold.dim_customer
UNION ALL
SELECT 
    'dim_product', COUNT(*), NULL FROM workspace.gold.dim_product
UNION ALL
SELECT 
    'dim_sellers', COUNT(*), NULL FROM workspace.gold.dim_sellers
UNION ALL
SELECT 
    'dim_date', COUNT(*), NULL FROM workspace.gold.dim_date;

In [0]:
%sql
-- Create dim_order_status
CREATE OR REPLACE TABLE workspace.gold.dim_order_status AS
SELECT DISTINCT order_status
FROM workspace.gold.fact_orders
WHERE order_status IS NOT NULL
ORDER BY order_status;


In [0]:
%sql
-- ============================================
-- VALIDATION TESTS
-- ============================================

-- TEST 1: All statuses in dim_order_status
SELECT '1. dim_order_status values' AS test, order_status, NULL AS cnt
FROM workspace.gold.dim_order_status

UNION ALL

-- TEST 2: Statuses in fact_orders NOT in dim
SELECT '2. Orphan in fact_orders', order_status, CAST(COUNT(*) AS STRING)
FROM workspace.gold.fact_orders
WHERE order_status NOT IN (SELECT order_status FROM workspace.gold.dim_order_status)
GROUP BY order_status

UNION ALL

-- TEST 3: Statuses in fact_payments NOT in dim
SELECT '3. Orphan in fact_payments', order_status, CAST(COUNT(*) AS STRING)
FROM workspace.gold.fact_payments
WHERE order_status NOT IN (SELECT order_status FROM workspace.gold.dim_order_status)
GROUP BY order_status

UNION ALL

-- TEST 4: NULL order_status in fact_orders
SELECT '4. NULL status fact_orders', 'null_count', CAST(COUNT(*) AS STRING)
FROM workspace.gold.fact_orders WHERE order_status IS NULL

UNION ALL

-- TEST 5: NULL order_status in fact_payments
SELECT '5. NULL status fact_payments', 'null_count', CAST(COUNT(*) AS STRING)
FROM workspace.gold.fact_payments WHERE order_status IS NULL

ORDER BY test;

In [0]:
%sql
-- TEST 6: Payment totals by order_status
SELECT 
    COALESCE(o.order_status, p.order_status) AS order_status,
    o.order_count,
    o.item_revenue,
    p.payment_count,
    p.payment_total
FROM (
    SELECT order_status,
           COUNT(DISTINCT order_id) AS order_count,
           ROUND(SUM(price), 2) AS item_revenue
    FROM workspace.gold.fact_orders
    GROUP BY order_status
) o
FULL OUTER JOIN (
    SELECT order_status,
           COUNT(*) AS payment_count,
           ROUND(SUM(payment_value), 2) AS payment_total
    FROM workspace.gold.fact_payments
    GROUP BY order_status
) p ON o.order_status = p.order_status
ORDER BY payment_total DESC;